# T-E2 AI Agent自动生成股市周报 — 数据清洗与整理

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI_Agent自动生成股市周报 |
| 小组   | 第 09 组 |
| 成员   | 梁志鹏（25210177）、吴璇子（25210264）、黄悦（25210145）、王鹤洋（25210243）、柯骋（25210150）、罗天（25210207）|
| GitHub | https://github.com/liangzhipeng82138-tech/ds2026-G09-ai_agent_stock_weekly_report |
| Pages  | https://liangzhipeng82138-tech.github.io/ds2026-G09-ai_agent_stock_weekly_report/ |
| 日期   | 2026-05-15 |

## 任务说明

本 Notebook 负责对 `data_raw/` 中的原始数据进行清洗和整理：
1. 处理缺失值和异常值
2. 统一日期格式和字段命名
3. 计算周涨跌幅、振幅等衍生指标
4. 汇总为结构化的周度数据
5. 保存清洗后数据至 `data_clean/` 目录

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime, timedelta

os.makedirs('../data_clean', exist_ok=True)
print('data_clean 目录已就绪')

### 1. 加载原始指数数据

In [ ]:
import glob

# 加载所有指数原始数据
index_raw = {}
for f in sorted(glob.glob('../data_raw/index_*.csv')):
    name = os.path.basename(f).replace('index_', '').replace('.csv', '')
    df = pd.read_csv(f)
    index_raw[name] = df
    print(f'已加载: {name} ({len(df)} 行)')
    print(f'  列: {list(df.columns)}')

### 2. 清洗指数数据

- 统一列名为英文
- 确保日期为 datetime 类型
- 检查缺失值
- 按日期排序

In [ ]:
index_clean = {}

for name, df in index_raw.items():
    # 统一列名
    col_map = {}
    for col in df.columns:
        col_lower = col.lower().strip()
        if 'date' in col_lower or '日期' in col_lower:
            col_map[col] = 'date'
        elif 'open' in col_lower or '开盘' in col_lower:
            col_map[col] = 'open'
        elif 'high' in col_lower or '最高' in col_lower:
            col_map[col] = 'high'
        elif 'low' in col_lower or '最低' in col_lower:
            col_map[col] = 'low'
        elif 'close' in col_lower or '收盘' in col_lower:
            col_map[col] = 'close'
        elif 'volume' in col_lower or '成交' in col_lower:
            col_map[col] = 'volume'
    
    df = df.rename(columns=col_map)
    
    # 日期格式转换
    df['date'] = pd.to_datetime(df['date'])
    
    # 排序
    df = df.sort_values('date').reset_index(drop=True)
    
    # 检查缺失值
    missing = df.isnull().sum()
    if missing.any():
        print(f'{name} 缺失值:')
        print(missing[missing > 0])
        # 前向填充
        df = df.fillna(method='ffill')
    
    index_clean[name] = df
    print(f'{name}: 清洗完成, {len(df)} 行, 列={list(df.columns)}')

print(f'\n共清洗 {len(index_clean)} 个指数数据集')

### 3. 计算周度衍生指标

对每个指数计算本周关键指标：
- 本周收盘价、上周收盘价
- 周涨跌幅
- 周振幅 = (最高 - 最低) / 上周收盘 * 100
- 周成交额

In [ ]:
# 计算周度指标
weekly_metrics = []

for name, df in index_clean.items():
    if len(df) < 6:
        print(f'{name}: 数据不足，跳过')
        continue
    
    # 取最近5个交易日作为本周
    this_week = df.tail(5)
    # 上周最后一天 = 本周第一天的前一天
    last_week_end_idx = df.index[-6]
    last_week_close = df.loc[last_week_end_idx, 'close']
    
    this_week_close = this_week['close'].iloc[-1]
    this_week_high = this_week['high'].max()
    this_week_low = this_week['low'].min()
    this_week_open = this_week['open'].iloc[0]
    
    # 周涨跌幅
    change_pct = (this_week_close / last_week_close - 1) * 100
    
    # 振幅
    amplitude = (this_week_high - this_week_low) / last_week_close * 100
    
    # 成交额（取最近5日之和）
    if 'volume' in this_week.columns:
        weekly_volume = this_week['volume'].sum()
    else:
        weekly_volume = np.nan
    
    metrics = {
        '指数': name,
        '本周收盘': round(this_week_close, 2),
        '上周收盘': round(last_week_close, 2),
        '周涨跌幅(%)': round(change_pct, 2),
        '振幅(%)': round(amplitude, 2),
        '周成交额': weekly_volume,
        '方向': '上涨' if change_pct >= 0 else '下跌',
    }
    weekly_metrics.append(metrics)
    
    print(f"{name}: 收盘={this_week_close:.2f}, 涨跌幅={change_pct:+.2f}%, 振幅={amplitude:.2f}%")

df_weekly = pd.DataFrame(weekly_metrics)
print(f'\n周度指标汇总:')
print(df_weekly.to_string(index=False))

### 4. 清洗全市场涨跌家数数据

In [ ]:
# 加载全市场行情原始数据
spot_path = '../data_raw/stock_spot_all.csv'
if os.path.exists(spot_path):
    spot_df = pd.read_csv(spot_path)
    print(f'全市场行情: {len(spot_df)} 只股票')
    print(f'列名: {list(spot_df.columns)}')
    
    # 根据涨跌幅统计涨跌家数
    # 尝试找到涨跌幅列
    change_col = None
    for col in spot_df.columns:
        if '涨跌幅' in str(col) or 'changepercent' in str(col).lower() or '涨跌' in str(col):
            change_col = col
            break
    
    if change_col:
        # 清洗：去除停牌股票（涨跌幅为NaN）
        spot_valid = spot_df.dropna(subset=[change_col])
        
        rise_count = (spot_valid[change_col] > 0).sum()
        fall_count = (spot_valid[change_col] < 0).sum()
        flat_count = (spot_valid[change_col] == 0).sum()
        
        # 涨停跌停统计
        limit_up = (spot_valid[change_col] >= 9.9).sum()   # 主板涨停
        limit_down = (spot_valid[change_col] <= -9.9).sum()
        
        # 科创板20%涨跌幅
        limit_up_20 = (spot_valid[change_col] >= 19.9).sum()
        limit_down_20 = (spot_valid[change_col] <= -19.9).sum()
        
        sentiment = {
            '上涨家数': int(rise_count),
            '下跌家数': int(fall_count),
            '平盘家数': int(flat_count),
            '涨停家数': int(limit_up + limit_up_20),
            '跌停家数': int(limit_down + limit_down_20),
            '赚钱效应(%)': round(rise_count / len(spot_valid) * 100, 1),
        }
        
        print(f'\n市场情绪统计:')
        for k, v in sentiment.items():
            print(f'  {k}: {v}')
        
        # 保存
        pd.DataFrame([sentiment]).to_csv('../data_clean/sentiment.csv', index=False, encoding='utf-8-sig')
        print('\n已保存: ../data_clean/sentiment.csv')
    else:
        print(f'未找到涨跌幅列，可用列: {list(spot_df.columns)}')
else:
    print('未找到全市场行情数据文件')

### 5. 清洗北向资金数据

In [ ]:
# 加载北向资金原始数据
nb_path = '../data_raw/northbound_flow.csv'
if os.path.exists(nb_path):
    nb_df = pd.read_csv(nb_path)
    print(f'北向资金: {len(nb_df)} 条记录')
    print(f'列名: {list(nb_df.columns)}')
    
    # 统一列名
    for col in nb_df.columns:
        if 'date' in str(col).lower() or '日期' in str(col):
            nb_df = nb_df.rename(columns={col: 'date'})
        elif '净流入' in str(col) or 'net' in str(col).lower():
            nb_df = nb_df.rename(columns={col: 'net_flow'})
    
    nb_df['date'] = pd.to_datetime(nb_df['date'])
    nb_df = nb_df.sort_values('date').reset_index(drop=True)
    
    # 填充缺失值
    nb_df = nb_df.fillna(0)
    
    # 取最近5个交易日
    nb_weekly = nb_df.tail(5).copy()
    
    # 计算本周净流入
    if 'net_flow' in nb_weekly.columns:
        total_net_flow = nb_weekly['net_flow'].sum()
        print(f'本周北向资金净流入: {total_net_flow:.2f} 亿元')
    
    nb_weekly.to_csv('../data_clean/northbound_weekly.csv', index=False, encoding='utf-8-sig')
    print('已保存: ../data_clean/northbound_weekly.csv')
else:
    print('未找到北向资金数据文件')

### 6. 汇总保存清洗后数据

In [ ]:
# 保存周度指标
df_weekly.to_csv('../data_clean/weekly_index_metrics.csv', index=False, encoding='utf-8-sig')
print('已保存: ../data_clean/weekly_index_metrics.csv')

# 保存清洗后的指数数据
for name, df in index_clean.items():
    df.to_csv(f'../data_clean/index_{name}_clean.csv', index=False, encoding='utf-8-sig')
    print(f'已保存: ../data_clean/index_{name}_clean.csv')

# 生成 JSON 格式周报数据（供 AI 和模板使用）
weekly_report_data = {
    'report_date': datetime.now().strftime('%Y-%m-%d'),
    'period': f'{(datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")} 至 {datetime.now().strftime("%Y-%m-%d")}',
    'indices': df_weekly.to_dict(orient='records'),
}

with open('../data_clean/weekly_data.json', 'w', encoding='utf-8') as f:
    json.dump(weekly_report_data, f, ensure_ascii=False, indent=2, default=str)
print('已保存: ../data_clean/weekly_data.json')

print('\n✓ 数据清洗完成！')